# 🤖 AI Travel Claim Assistant: User Dashboard
Welcome to your simplified processing panel. Follow the steps below to audit and compile your travel expenses automatically into standard corporate system import files.

### 📝 Step 1: First-Time Setup Only
If this is your first time using the assistant, you must link your secure AI key:
1. Click the **Key Icon (🔑)** on the far left vertical sidebar to open the **Secrets Panel**.
2. Click **"Add new secret"** and type standard configuration markers:
   - **Name:** `GEMINI_API_KEY` (Must be all uppercase)
   - **Value:** Paste your secure token text string (starting with `AIzaSy...` generated from Google AI Studio).
3. **Crucial:** Flip the toggle switch to **On** (*"Grant Notebook Access"*).

### 📁 Step 2: Organize Your Trip Receipts
1. Open your **Google Drive**.
2. Navigate into your auto-generated workspace folder: `Travel_Claims_Workspace / 1_DROP_RECEIPTS_HERE /`.
3. Create a **new folder** named exactly after your Unique Trip or Claim Tracking ID (e.g., `CLAIM-2026-001`).
4. Drop all invoices, digital receipts, or text expense logs from that trip directly into that folder.

### 🚀 Step 3: Configure and Execute Audit
Update the tracking ID text string inside the quotation marks below to match your folder name exactly, then go to the top menu toolbar and click **Runtime ➔ Run all** to process your claim report!

In [ ]:
#@title ⚙️ ACTIVE CLAIM ID MANAGEMENT
# Paste your exact target folder string name inside the quotation marks:
ACTIVE_CLAIM_ID = "CLAIM-2026-001" 

# =====================================================================
# CORE PRODUCTION PROCESSING PIPELINE ENGINE (AUTOMATED RUNTIME)
# =====================================================================
print("📦 Synchronizing system environment dependencies...")
!pip install -q -U google-genai

from google.colab import drive, userdata
from google.genai import Client
import pandas as pd
import json
import time
import os

print("🔄 Initializing processing runtime. Mounting storage paths...")
drive.mount('/content/drive', force_remount=True)

BASE_WORKSPACE = "/content/drive/MyDrive/Travel_Claims_Workspace"
INPUT_DIR = f"{{BASE_WORKSPACE}}/1_DROP_RECEIPTS_HERE/{{ACTIVE_CLAIM_ID}}"
OUTPUT_DIR = f"{{BASE_WORKSPACE}}/2_GENERATED_SYSTEM_REPORTS"

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = Client(api_key=userdata.get('GEMINI_API_KEY'))

if os.path.exists(INPUT_DIR):
    files = [f for f in os.listdir(INPUT_DIR) if os.path.isfile(os.path.join(INPUT_DIR, f))]
    
    if not files:
        print(f"🛑 Pipeline Halted: No transaction documents found inside folder: {{INPUT_DIR}}")
    else:
        print(f"Executing processing run for Claim ID: {{ACTIVE_CLAIM_ID}} ({{len(files)}} items discovered)")
        parsed_records = []
        
        for file_name in files:
            with open(os.path.join(INPUT_DIR, file_name), "r") as f:
                raw_text = f.read()
                
            system_prompt = f"""
            You are a backend database parser. Extract values from this receipt text.
            You MUST respond ONLY with a valid, clean JSON object matching this exact schema:
            {{
                \"VendorName\": \"String\",
                \"TransactionTotal\": Float,
                \"ApprovedAmount\": Float,
                \"TaxPaid\": Float,
                \"PolicyViolationFlag\": \"Y\" or \"N\",
                \"AuditComments\": \"String\"
            }}
            Raw text:
            {{raw_text}}
            """
            
            for attempt in range(1, 4):
                try:
                    response = client.models.generate_content(model='gemini-3.8-flash', contents=system_prompt)
                    clean_json = response.text.replace("```json", "").replace("```", "").strip()
                    item_data = json.loads(clean_json)
                    item_data["ClaimID"] = ACTIVE_CLAIM_ID
                    item_data["SourceFileName"] = file_name
                    parsed_records.append(item_data)
                    print(f"   -> Successfully processed transaction entry: {{file_name}}")
                    break
                except Exception:
                    time.sleep(3)
            time.sleep(1.5)
            
        if parsed_records:
            df = pd.DataFrame(parsed_records)
            column_order = ["ClaimID", "VendorName", "TransactionTotal", "ApprovedAmount", "TaxPaid", "PolicyViolationFlag", "SourceFileName", "AuditComments"]
            df = df[column_order]
            
            csv_filename = f"{{OUTPUT_DIR}}/{{ACTIVE_CLAIM_ID}}_IMPORT.csv"
            df.to_csv(csv_filename, index=False)
            
            md_filename = f"{{OUTPUT_DIR}}/{{ACTIVE_CLAIM_ID}}_SUMMARY.md"
            with open(md_filename, "w") as md:
                md.write(f"# Human-Readable Review: {{ACTIVE_CLAIM_ID}}\n\n")
                md.write(f"Total Line Items Processed: {{len(parsed_records)}}\n")
                md.write(f"Total Requested: ${{df['TransactionTotal'].sum():.2f}}\n")
                md.write(f"**Total Approved for System Import:** **${{df['ApprovedAmount'].sum():.2f}}**\n\n")
                md.write(df.to_markdown(index=False))
                
            print("\n" + "="*65)
            print(f"🎉 AUDIT PROCESS COMPLETE FOR DATA SET: {{ACTIVE_CLAIM_ID}}")
            print(f"💾 Claims System Import Sheet Ready: {{csv_filename}}")
            print(f"📄 Approval Overview Summary Ready: {{md_filename}}")
            print("="*65)
else:
    print(f"🛑 System Error: Target path folder path missing: {{INPUT_DIR}}")